In [1]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, IFrame, display


# ---------------------------------------------------------------------
# Repository paths
# ---------------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "run_privacy_scenario.py").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "run_privacy_scenario.py").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate run_privacy_scenario.py. "
        "Run this notebook from 3-privacy-coordination-patterns/ "
        "or 3-privacy-coordination-patterns/analysis/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import run_privacy_scenario as scenario


# ---------------------------------------------------------------------
# Input data
# ---------------------------------------------------------------------

SUMMARY_MODE = "from_plots"

summary_paths = {
    "from_plots": scenario.RESULTS_DIR / "privacy_scenario_summary_from_plots.csv",
    "current_csv": scenario.RESULTS_DIR / "privacy_scenario_summary.csv",
}

summary_path = summary_paths[SUMMARY_MODE]

if not summary_path.exists():
    raise FileNotFoundError(f"Summary file not found: {summary_path}")

summary = pd.read_csv(summary_path)


required_columns = {
    "workload_label",
    "privacy_label",
    "architecture_label",
    "avg_latency_s",
    "avg_success_rate",
    "avg_privacy_comm_burden_s",
}

missing_columns = sorted(required_columns - set(summary.columns))

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")


# ---------------------------------------------------------------------
# Experiment configuration
# ---------------------------------------------------------------------

privacy_order = list(scenario.PRIVACY_LEVELS.keys())
workload_order = list(scenario.WORKLOAD_LEVELS.keys())
arch_order = list(scenario.ARCHITECTURES.keys())

arch_labels = {
    "Centralized": "CE",
    "Semi-Decentralized": "SD",
    "Decentralized": "FD",
}

colors = {
    "Decentralized": "#666666",
    "Semi-Decentralized": "#999999",
    "Centralized": "#b3b3b3",
}

privacy_tick_labels = ["Low", "Medium", "High"]


# ---------------------------------------------------------------------
# Results table
# ---------------------------------------------------------------------

table_rows = []

for workload in workload_order:
    for privacy_idx, privacy in enumerate(privacy_order):

        subset = summary[
            (summary["workload_label"] == workload)
            & (summary["privacy_label"] == privacy)
        ]

        best_latency = subset["avg_latency_s"].min()
        best_throughput = subset["avg_success_rate"].max()

        workload_cell = workload.replace(" Load", "") if privacy_idx == 1 else ""
        privacy_cell = privacy.replace(" Privacy Overhead", "")

        latency_cells = []
        throughput_cells = []

        for arch in arch_order:

            row = subset[
                subset["architecture_label"] == arch
            ].iloc[0]

            latency = float(row["avg_latency_s"])
            throughput = float(row["avg_success_rate"]) * 100

            latency_text = f"{latency:.2f}"
            throughput_text = f"{throughput:.2f}"

            if latency == best_latency:
                latency_text = f"<strong>{latency_text}</strong>"

            if float(row["avg_success_rate"]) == best_throughput:
                throughput_text = f"<strong>{throughput_text}</strong>"

            latency_cells.append(latency_text)
            throughput_cells.append(throughput_text)

        table_rows.append(
            "<tr>"
            f"<td>{workload_cell}</td>"
            f"<td>{privacy_cell}</td>"
            + "".join(f"<td>{value}</td>" for value in latency_cells)
            + "".join(f"<td>{value}</td>" for value in throughput_cells)
            + "</tr>"
        )


table_html = """
<table style="border-collapse:collapse; font-size:12px; margin-bottom:16px; text-align:right;">
  <thead>
    <tr>
      <th rowspan="2" style="padding:4px 8px; text-align:left; border-bottom:1px solid #222;">Workload</th>
      <th rowspan="2" style="padding:4px 8px; text-align:left; border-bottom:1px solid #222;">Privacy</th>
      <th colspan="3" style="padding:4px 8px; text-align:center; border-bottom:1px solid #222;">Latency (sec.)</th>
      <th colspan="3" style="padding:4px 8px; text-align:center; border-bottom:1px solid #222;">Throughput (%)</th>
    </tr>
    <tr>
      <th style="padding:4px 8px;">CE</th>
      <th style="padding:4px 8px;">SD</th>
      <th style="padding:4px 8px;">FD</th>
      <th style="padding:4px 8px;">CE</th>
      <th style="padding:4px 8px;">SD</th>
      <th style="padding:4px 8px;">FD</th>
    </tr>
  </thead>
  <tbody>
""" + "\n".join(table_rows) + """
  </tbody>
</table>
"""

display(HTML(table_html))


# ---------------------------------------------------------------------
# Plots
# ---------------------------------------------------------------------

plot_specs = [
    ("avg_latency_s", "Average Latency (sec.)", "latency_by_privacy.pdf"),
    ("avg_success_rate", "Throughput (%)", "success_rate_by_privacy.pdf"),
    (
        "avg_privacy_comm_burden_s",
        "Privacy Overhead (sec.)",
        "privacy_burden_by_privacy.pdf",
    ),
]

# Ensure output directory exists
scenario.PLOTS_DIR.mkdir(parents=True, exist_ok=True)


for plot_idx, (metric, ylabel, filename) in enumerate(plot_specs):

    fig, axes = plt.subplots(
        1,
        len(workload_order),
        figsize=(
            scenario.FIGSIZE[0] * len(workload_order),
            scenario.FIGSIZE[1],
        ),
        sharey=False,
    )

    if len(workload_order) == 1:
        axes = [axes]

    for ax, workload in zip(axes, workload_order):

        subset = summary[
            summary["workload_label"] == workload
        ].copy()

        x = range(len(privacy_order))
        width = 0.23

        ax.set_axisbelow(True)
        ax.grid(
            axis="y",
            linestyle=":",
            linewidth=0.5,
        )

        for idx, arch in enumerate(arch_order):

            vals = []

            for privacy in privacy_order:

                row = subset[
                    (subset["privacy_label"] == privacy)
                    & (subset["architecture_label"] == arch)
                ]

                value = float(row[metric].iloc[0])

                if metric == "avg_success_rate":
                    value *= 100

                vals.append(value)

            ax.bar(
                [i + (idx - 1) * width for i in x],
                vals,
                width=width,
                label=arch_labels[arch],
                color=colors[arch],
                edgecolor="black",
                linewidth=0.4,
            )

        ax.set_title(
            workload.replace("Load", "Workload")
        )

        ax.set_xticks(list(x))
        ax.set_xticklabels(privacy_tick_labels)
        ax.set_xlabel("Privacy Overhead")
        ax.tick_params(axis="both", labelsize=8)

        if metric == "avg_success_rate":
            ax.set_ylim(90, 100)

    axes[0].set_ylabel(ylabel)

    if plot_idx == 0:

        handles, labels = axes[0].get_legend_handles_labels()

        fig.legend(
            handles,
            labels,
            title="Coordination Mechanisms",
            loc="lower center",
            bbox_to_anchor=(0.5, 1.02),
            ncol=len(arch_order),
            frameon=False,
        )

    fig.tight_layout()

    pdf_path = scenario.PLOTS_DIR / filename

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.close(fig)

    print(f"Saved: {pdf_path}")

    display(
        IFrame(
            src=str(pdf_path.resolve()),
            width="100%",
            height=320,
        )
    )

Saved: /Users/yelyzavetakurkchi/Documents/GSSI/Thesis/Replication_Package/thesis-replication-package/3-privacy-coordination-patterns/results/plots/latency_by_privacy.pdf


Saved: /Users/yelyzavetakurkchi/Documents/GSSI/Thesis/Replication_Package/thesis-replication-package/3-privacy-coordination-patterns/results/plots/success_rate_by_privacy.pdf


Saved: /Users/yelyzavetakurkchi/Documents/GSSI/Thesis/Replication_Package/thesis-replication-package/3-privacy-coordination-patterns/results/plots/privacy_burden_by_privacy.pdf
